# Day 7 — DuckDB: Fast Local SQL Analytics
**Date:** 28 March 2026 | **Phase 1 — Week 1** | **Roadmap: DS-AI-75D**

> DuckDB — run SQL directly on DataFrames and CSV files.
> No server, no setup, 10x faster than SQLite for analytics.

## 1. Setup & Why DuckDB

In [ ]:
import duckdb
import pandas as pd
import numpy as np

print(f"DuckDB version: {duckdb.__version__}")

# Connect to an in-memory database — no files needed
con = duckdb.connect()

# Load Titanic data
df = pd.read_csv(r"C:\DS-AI-75d\titanic.csv")
print(f"Dataset: {df.shape}")
print("DuckDB ready!")

DuckDB version: 1.5.1
Dataset: (891, 12)
DuckDB ready!


## 2. DuckDB Fundamentals — SQL on DataFrames

### 2.1 Querying a Pandas DataFrame Directly with SQL

In [ ]:
# DuckDB can query Pandas DataFrames directly — no loading needed
# Just need to reference the variable name in the SQL query

result = con.execute(
    """
    SELECT
        Pclass,
        COUNT(*) AS total_passengers,
        SUM(Survived) AS survivors,
        ROUND(AVG(Survived) * 100, 1) AS survival_rate_pct,
        ROUND(AVG(Age), 1) AS avg_age,
        ROUND(AVG(Fare), 2) AS avg_fare
    FROM df
    GROUP BY Pclass
    ORDER BY Pclass
"""
).df()  # .df() converts result to Pandas DataFrame

print("Survival stats by class:")
print(result)

Survival stats by class:
   Pclass  total_passengers  survivors  survival_rate_pct  avg_age  avg_fare
0       1               216      136.0               63.0     38.2     84.15
1       2               184       87.0               47.3     29.9     20.66
2       3               491      119.0               24.2     25.1     13.68


### 2.2 Filtering and Selecting with SQL

In [ ]:
# SQL filtering — same as Pandas boolean indexing but in SQL
first_class_women = con.execute(
    """
    SELECT
        Name,
        Age,
        Fare,
        Survived
    FROM df
    WHERE Pclass = 1
      AND Sex = 'female'
      AND Age IS NOT NULL
    ORDER BY Fare DESC
    LIMIT 10
"""
).df()

print("Top 10 first class women by fare:")
print(first_class_women)

Top 10 first class women by fare:
                                                Name   Age      Fare  Survived
0                                   Ward, Miss. Anna  35.0  512.3292         1
1                     Fortune, Miss. Alice Elizabeth  24.0  263.0000         1
2                         Fortune, Miss. Mabel Helen  23.0  263.0000         1
3                         Ryerson, Miss. Emily Borie  18.0  262.3750         1
4              Ryerson, Miss. Susan Parker "Suzette"  21.0  262.3750         1
5    Baxter, Mrs. James (Helene DeLaudeniere Chaput)  50.0  247.5208         1
6                              Bidois, Miss. Rosalie  42.0  227.5250         1
7                      Endres, Miss. Caroline Louise  38.0  227.5250         1
8  Astor, Mrs. John Jacob (Madeleine Talmadge Force)  18.0  227.5250         1
9                      Allen, Miss. Elisabeth Walton  29.0  211.3375         1


### 2.3 Querying CSV Files Directly — No Loading Required

In [ ]:
# DuckDB can query CSV files DIRECTLY — no pd.read_csv() needed
# This is one of DuckDB's most powerful features for large files

result = con.execute(
    """
    SELECT
        Sex,
        COUNT(*) AS total,
        ROUND(AVG(Survived) * 100, 1) AS survival_rate_pct,
        ROUND(AVG(Age), 1) AS avg_age
    FROM read_csv_auto('C:/DS-AI-75d/titanic.csv')
    WHERE Age IS NOT NULL
    GROUP BY Sex
    ORDER BY survival_rate_pct DESC
"""
).df()

print("Survival stats by gender (queried directly from CSV):")
print(result)

Survival stats by gender (queried directly from CSV):
      Sex  total  survival_rate_pct  avg_age
0  female    261               75.5     27.9
1    male    453               20.5     30.7


### 2.4 SQL Joins in DuckDB

In [ ]:
# Create two DataFrames to join
passengers = df[["PassengerId", "Name", "Pclass", "Survived"]].copy()

class_info = pd.DataFrame(
    {
        "Pclass": [1, 2, 3],
        "ClassName": ["First Class", "Second Class", "Third Class"],
        "TicketPrice": ["Expensive", "Moderate", "Cheap"],
    }
)

# SQL JOIN — exactly like database joins
result = con.execute(
    """
    SELECT
        p.Name,
        p.Pclass,
        c.ClassName,
        c.TicketPrice,
        p.Survived
    FROM passengers p
    JOIN class_info c ON p.Pclass = c.Pclass
    WHERE p.Survived = 1
    ORDER BY p.Pclass
    LIMIT 8
"""
).df()

print("Survivors with class information:")
print(result)

Survivors with class information:
                                                Name  Pclass    ClassName  \
0                           Bonnell, Miss. Elizabeth       1  First Class   
1                       Sloper, Mr. William Thompson       1  First Class   
2  Cumings, Mrs. John Bradley (Florence Briggs Th...       1  First Class   
3     Spencer, Mrs. William Augustus (Marie Eugenie)       1  First Class   
4           Harper, Mrs. Henry Sleeper (Myna Haxtun)       1  First Class   
5       Futrelle, Mrs. Jacques Heath (Lily May Peel)       1  First Class   
6                                  Woolner, Mr. Hugh       1  First Class   
7                                Icard, Miss. Amelie       1  First Class   

  TicketPrice  Survived  
0   Expensive         1  
1   Expensive         1  
2   Expensive         1  
3   Expensive         1  
4   Expensive         1  
5   Expensive         1  
6   Expensive         1  
7   Expensive         1  


## 3. DuckDB Advanced — Window Functions & CTEs

### 3.1 Window Functions — Rank Within Groups

In [ ]:
# Window functions — calculate rankings and running totals
# RANK() OVER (PARTITION BY ... ORDER BY ...) is powerful analytics SQL

result = con.execute(
    """
    SELECT
        Name,
        Pclass,
        Fare,
        RANK() OVER (
            PARTITION BY Pclass
            ORDER BY Fare DESC
        ) AS fare_rank_in_class,
        ROUND(AVG(Fare) OVER (PARTITION BY Pclass), 2) AS class_avg_fare,
        ROUND(Fare - AVG(Fare) OVER (PARTITION BY Pclass), 2) AS diff_from_avg
    FROM df
    WHERE Fare IS NOT NULL
    ORDER BY Pclass, fare_rank_in_class
    LIMIT 12
"""
).df()

print("Fare rankings within each class:")
print(result)

Fare rankings within each class:
                                               Name  Pclass      Fare  \
0                Cardeza, Mr. Thomas Drake Martinez       1  512.3292   
1                            Lesurer, Mr. Gustave J       1  512.3292   
2                                  Ward, Miss. Anna       1  512.3292   
3                    Fortune, Miss. Alice Elizabeth       1  263.0000   
4                    Fortune, Mr. Charles Alexander       1  263.0000   
5                        Fortune, Miss. Mabel Helen       1  263.0000   
6                                 Fortune, Mr. Mark       1  263.0000   
7             Ryerson, Miss. Susan Parker "Suzette"       1  262.3750   
8                        Ryerson, Miss. Emily Borie       1  262.3750   
9                          Baxter, Mr. Quigg Edmond       1  247.5208   
10  Baxter, Mrs. James (Helene DeLaudeniere Chaput)       1  247.5208   
11                              Robbins, Mr. Victor       1  227.5250   

    fare_rank_in_

### 3.2 CTEs — Common Table Expressions

In [ ]:
# CTEs make complex queries readable — like named subqueries
# WITH cte_name AS (...) SELECT ... FROM cte_name

result = con.execute(
    """
    WITH survival_stats AS (
        SELECT
            Pclass,
            Sex,
            COUNT(*) AS total,
            SUM(Survived) AS survivors,
            ROUND(AVG(Survived) * 100, 1) AS survival_pct
        FROM df
        GROUP BY Pclass, Sex
    ),
    class_totals AS (
        SELECT
            Pclass,
            SUM(total) AS class_total,
            ROUND(AVG(survival_pct), 1) AS class_avg_survival
        FROM survival_stats
        GROUP BY Pclass
    )
    SELECT
        s.Pclass,
        s.Sex,
        s.total,
        s.survival_pct,
        c.class_avg_survival,
        ROUND(s.survival_pct - c.class_avg_survival, 1) AS diff_from_class_avg
    FROM survival_stats s
    JOIN class_totals c ON s.Pclass = c.Pclass
    ORDER BY s.Pclass, s.Sex
"""
).df()

print("Survival analysis with CTEs:")
print(result)

Survival analysis with CTEs:
   Pclass     Sex  total  survival_pct  class_avg_survival  \
0       1  female     94          96.8                66.9   
1       1    male    122          36.9                66.9   
2       2  female     76          92.1                53.9   
3       2    male    108          15.7                53.9   
4       3  female    144          50.0                31.8   
5       3    male    347          13.5                31.8   

   diff_from_class_avg  
0                 29.9  
1                -30.0  
2                 38.2  
3                -38.2  
4                 18.2  
5                -18.3  


### 3.3 DuckDB vs Pandas — Speed Comparison

In [ ]:
import time

# Generate large dataset — 1 million rows
np.random.seed(42)
n = 1_000_000
large_df = pd.DataFrame(
    {
        "id": np.arange(n),
        "category": np.random.choice(["A", "B", "C", "D"], n),
        "value": np.random.randn(n),
        "score": np.random.uniform(0, 100, n),
    }
)

# Pandas GroupBy
start = time.time()
pandas_result = large_df.groupby("category")["score"].agg(["mean", "sum", "count"])
pandas_time = time.time() - start
print(f"Pandas GroupBy:  {pandas_time:.4f} seconds")

# DuckDB SQL GroupBy
start = time.time()
duckdb_result = con.execute(
    """
    SELECT
        category,
        AVG(score) AS mean,
        SUM(score) AS sum,
        COUNT(*) AS count
    FROM large_df
    GROUP BY category
    ORDER BY category
"""
).df()
duckdb_time = time.time() - start
print(f"DuckDB GroupBy:  {duckdb_time:.4f} seconds")
print(f"\nDuckDB is {pandas_time/duckdb_time:.1f}x faster than Pandas!")
print("\nDuckDB result:")
print(duckdb_result)

Pandas GroupBy:  0.1304 seconds
DuckDB GroupBy:  0.0291 seconds

DuckDB is 4.5x faster than Pandas!

DuckDB result:
  category       mean           sum   count
0        A  50.028456  1.251807e+07  250219
1        B  50.016119  1.249173e+07  249754
2        C  49.950365  1.248609e+07  249970
3        D  49.945894  1.248932e+07  250057


## 4. Key Insights — DuckDB

1. **Query DataFrames with SQL** — reference any Pandas variable directly in SQL
2. **read_csv_auto()** — query CSV files without loading into memory first
3. **Window functions** — RANK(), AVG() OVER PARTITION BY in pure SQL
4. **CTEs** — WITH clause makes complex multi-step queries readable
5. **4.5x faster than Pandas** on GroupBy — uses vectorised C++ execution
6. **No server needed** — runs entirely in-process, zero setup
7. **Best use case** — complex multi-table SQL analytics on local data up to ~100GB